# Density rescue EDA for unresolved v4 ingredient lines

Goal: investigate whether the FAO/INFOODS density table in `Data/conversions/food_density.csv` can augment the current USDA portion/FDC pipeline and turn some unresolved ingredient lines into gram-resolvable rows.

This notebook is intentionally local-only. It reads the emailed error-analysis CSV already present at `scratch/EDA/portion_feasibility_1000/v4_unresolved_fdc_or_grams.csv` and does not write to the database or call external services.

## Pipeline context

Current v4 resolution requires both:

- a real USDA/FDC match (`llm_fdc_id` present and not sentinel `999000001`), and
- a non-null `grams` value.

`missing_fdc` rows are not fully rescued by density alone because nutrition lookup still needs a real FDC id. Density can still estimate grams for volume lines if an FDC match is later supplied.

`no_portion` rows are the main density-rescue target: they already have a real FDC id, but USDA `food_portion` did not provide a usable volume/count portion. For volume amounts, density can convert `quantity * unit -> ml -> grams`. For count amounts, density generally cannot help unless the count unit also implies a volume/container size, so count rescue should be treated as a separate and more manual problem.

Relevant code paths:

- `scripts/ingredient_match_llm_portion.py`: portion-aware FDC judge and initial gram metadata.
- `scripts/portion_gram.py`: gram-resolution ladder; terminal statuses include `missing_fdc`, `no_portion`, `vague_amount`, `unresolvable_serving_only`, `ambiguous_accepted`, and `bad_unit`.
- `scripts/portion_pipeline_feasibility.py`: orchestration and optional portion-LLM rescue.
- `scripts/resolved_recipe_portion.py`: MVP fully resolved recipe filtering.

In [ ]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "unit_convert.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find repo root containing scripts/unit_convert.py. "
        "Open this notebook from the Capstone repo or set ROOT manually."
    )

ROOT = find_repo_root()

sys.path.insert(0, str(ROOT / "scripts"))

from unit_convert import UnitConversionError, convert_volume, normalize_volume_unit

try:
    from rapidfuzz import fuzz, process
    HAVE_RAPIDFUZZ = True
except ImportError:
    from difflib import SequenceMatcher
    HAVE_RAPIDFUZZ = False

SENTINEL = 999_000_001
UNRESOLVED_CSV = ROOT / "scratch" / "EDA" / "portion_feasibility_1000" / "v4_unresolved_fdc_or_grams.csv"
DENSITY_CSV = ROOT / "Data" / "conversions" / "food_density.csv"
OUT_DIR = ROOT / "scratch" / "EDA" / "density_unresolved"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 180)

print(UNRESOLVED_CSV)
print(DENSITY_CSV)
print(f"rapidfuzz available: {HAVE_RAPIDFUZZ}")

In [ ]:
unresolved = pd.read_csv(UNRESOLVED_CSV)
density = pd.read_csv(DENSITY_CSV)

unresolved["llm_fdc_id_num"] = pd.to_numeric(unresolved["llm_fdc_id"], errors="coerce")
unresolved["has_real_fdc"] = unresolved["llm_fdc_id_num"].notna() & (unresolved["llm_fdc_id_num"].astype("Int64") != SENTINEL)
unresolved["has_grams"] = pd.to_numeric(unresolved["grams"], errors="coerce").notna()
unresolved["quantity_num"] = pd.to_numeric(unresolved["quantity"], errors="coerce")

def parse_density_value(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if "-" in text:
        parts = [pd.to_numeric(p.strip(), errors="coerce") for p in text.split("-", 1)]
        if all(pd.notna(p) for p in parts):
            return float(sum(parts) / 2)
    return pd.to_numeric(text, errors="coerce")

density["density_g_per_ml_raw"] = density["density_g_per_ml"]
density["density_g_per_ml"] = density["density_g_per_ml"].map(parse_density_value)
density = density[density["density_g_per_ml"].notna()].reset_index(drop=True).copy()

print(f"unresolved rows: {len(unresolved):,}")
print(f"density rows: {len(density):,}")
display(unresolved.head())
display(density.head())

In [ ]:
summary = (
    unresolved
    .groupby(["grams_status", "amount_kind_final"], dropna=False)
    .size()
    .rename("n")
    .reset_index()
    .sort_values(["grams_status", "n"], ascending=[True, False])
)

display(unresolved["grams_status"].value_counts(dropna=False).to_frame("n"))
display(unresolved["amount_kind_final"].value_counts(dropna=False).to_frame("n"))
display(summary)

target = unresolved[(unresolved["grams_status"].eq("no_portion")) & unresolved["has_real_fdc"]].copy()
print(f"Primary density-rescue target: {len(target):,} no_portion rows with real FDC")
display(target["amount_kind_final"].value_counts(dropna=False).to_frame("n"))

## Matching approach

Density table rows are generic food names, not FDC ids. This means a density match is evidence for a conversion factor, not a database join. The conservative workflow below:

1. Normalize ingredient names and density food names.
2. Fuzzy match each unresolved row to the best density row.
3. Flag high-confidence candidates using score and top-2 separation.
4. Convert volume rows to grams only when the recipe unit is supported.
5. Export likely/rescue-review tables for human inspection before changing the pipeline.

In [ ]:
STOPWORDS = {
    "fresh", "dried", "dry", "ground", "chopped", "chop", "minced", "diced", "sliced", "slice", "crushed",
    "large", "small", "medium", "extra", "fine", "finely", "coarse", "coarsely", "packed", "loosely",
    "optional", "to", "taste", "about", "plus", "more", "or", "as", "needed", "divided", "peeled", "seeded",
    "drained", "rinsed", "cooked", "uncooked", "raw", "prepared", "whole", "halved", "quartered",
}

UNIT_WORDS = {
    "cup", "cups", "c", "tablespoon", "tablespoons", "tbsp", "tbs", "teaspoon", "teaspoons", "tsp", "tsps",
    "pint", "pints", "pt", "quart", "quarts", "qt", "ml", "milliliter", "milliliters", "liter", "liters",
    "can", "cans", "jar", "jars", "bottle", "bottles", "package", "packages", "pkg", "box", "boxes",
    "ounce", "ounces", "oz", "pound", "pounds", "lb", "lbs", "gram", "grams", "g", "kg",
}

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\b\d+(?:\s+\d+\/\d+|\/\d+|\.\d+)?\b", " ", text)
    tokens = [t for t in text.split() if t not in STOPWORDS and t not in UNIT_WORDS]
    return " ".join(tokens)

def _fallback_similarity(a, b):
    a_tokens = set(a.split())
    b_tokens = set(b.split())
    token_overlap = 0.0 if not a_tokens or not b_tokens else len(a_tokens & b_tokens) / len(a_tokens | b_tokens)
    seq = SequenceMatcher(None, a, b).ratio()
    return 100 * max(seq, token_overlap)

def best_density_match(query, choices):
    if not query:
        return pd.Series({"density_match_idx": pd.NA, "density_match_score": 0.0, "density_second_score": 0.0})
    if HAVE_RAPIDFUZZ:
        matches = process.extract(query, choices, scorer=fuzz.WRatio, limit=2)
        best = matches[0] if matches else (None, 0, None)
        second = matches[1] if len(matches) > 1 else (None, 0, None)
    else:
        scored = sorted(
            ((choice, _fallback_similarity(query, choice), idx) for idx, choice in enumerate(choices)),
            key=lambda x: x[1],
            reverse=True,
        )[:2]
        best = scored[0] if scored else (None, 0, None)
        second = scored[1] if len(scored) > 1 else (None, 0, None)
    return pd.Series({
        "density_match_idx": best[2],
        "density_match_score": float(best[1]),
        "density_second_score": float(second[1]),
    })

density["density_name_norm"] = density["food_name"].map(normalize_text)
density_choices = density["density_name_norm"].fillna("").tolist()

unresolved["match_query"] = unresolved["name"].fillna(unresolved["ingredient"]).map(normalize_text)
match_cols = unresolved["match_query"].apply(lambda q: best_density_match(q, density_choices))
matched = pd.concat([unresolved.reset_index(drop=True), match_cols], axis=1)
matched["density_match_idx"] = pd.to_numeric(matched["density_match_idx"], errors="coerce")

density_lookup = density.reset_index().rename(columns={"index": "density_match_idx"})
matched = matched.merge(
    density_lookup[["density_match_idx", "food_group", "food_name", "density_g_per_ml", "density_name_norm"]],
    on="density_match_idx",
    how="left",
)
matched["density_score_gap"] = matched["density_match_score"] - matched["density_second_score"]

matched["density_confidence"] = np.select(
    [
        (matched["density_match_score"] >= 97) & (matched["density_score_gap"] >= 3),
        (matched["density_match_score"] >= 92) & (matched["density_score_gap"] >= 5),
        (matched["density_match_score"] >= 86),
    ],
    ["likely", "review", "weak"],
    default="reject",
)

display(matched[["ingredient", "name", "grams_status", "amount_kind_final", "match_query", "food_name", "density_g_per_ml", "density_match_score", "density_score_gap", "density_confidence"]].head(20))

In [ ]:
def volume_ml(quantity, unit):
    if pd.isna(quantity) or pd.isna(unit) or str(unit).strip() == "":
        return np.nan
    try:
        return convert_volume(float(quantity), str(unit), "milliliter")
    except (ValueError, UnitConversionError):
        return np.nan

matched["volume_ml"] = [volume_ml(q, u) for q, u in zip(matched["quantity_num"], matched["unit"])]
matched["density_estimated_grams"] = matched["volume_ml"] * matched["density_g_per_ml"]

matched["density_rescue_bucket"] = np.select(
    [
        matched["grams_status"].eq("no_portion") & matched["has_real_fdc"] & matched["amount_kind_final"].eq("volume") & matched["volume_ml"].notna() & matched["density_confidence"].eq("likely"),
        matched["grams_status"].eq("no_portion") & matched["has_real_fdc"] & matched["amount_kind_final"].eq("volume") & matched["volume_ml"].notna() & matched["density_confidence"].eq("review"),
        matched["grams_status"].eq("no_portion") & matched["has_real_fdc"] & matched["amount_kind_final"].eq("volume") & matched["volume_ml"].notna() & matched["density_confidence"].isin(["weak", "reject"]),
        matched["grams_status"].eq("missing_fdc") & matched["amount_kind_final"].eq("volume") & matched["volume_ml"].notna() & matched["density_confidence"].isin(["likely", "review"]),
        matched["amount_kind_final"].eq("count") & matched["density_confidence"].isin(["likely", "review"]),
    ],
    [
        "candidate_volume_rescue_likely",
        "candidate_volume_rescue_review",
        "volume_density_match_too_weak",
        "grams_possible_but_still_missing_fdc",
        "count_density_name_match_only",
    ],
    default="not_density_rescuable",
)

display(matched["density_rescue_bucket"].value_counts().to_frame("n"))
display(pd.crosstab(matched["grams_status"], matched["density_rescue_bucket"]))

In [ ]:
cols = [
    "recipe_id", "ingredient_idx", "ingredient", "quantity", "unit", "name", "amount_kind_final",
    "llm_fdc_id", "llm_description", "grams_status", "grams_method", "llm_certainty", "pipeline_path",
    "food_group", "food_name", "density_g_per_ml", "density_match_score", "density_second_score", "density_score_gap", "density_confidence",
    "volume_ml", "density_estimated_grams", "density_rescue_bucket",
]

likely_volume = matched[matched["density_rescue_bucket"].eq("candidate_volume_rescue_likely")].copy()
review_volume = matched[matched["density_rescue_bucket"].eq("candidate_volume_rescue_review")].copy()
missing_fdc_volume = matched[matched["density_rescue_bucket"].eq("grams_possible_but_still_missing_fdc")].copy()
count_name_only = matched[matched["density_rescue_bucket"].eq("count_density_name_match_only")].copy()

print(f"Likely direct volume rescues: {len(likely_volume):,}")
print(f"Review direct volume rescues: {len(review_volume):,}")
print(f"Volume grams possible but missing FDC: {len(missing_fdc_volume):,}")
print(f"Count rows with density name matches only: {len(count_name_only):,}")

display(likely_volume[cols].sort_values(["density_match_score", "density_score_gap"], ascending=False).head(50))
display(review_volume[cols].sort_values(["density_match_score", "density_score_gap"], ascending=False).head(50))

In [ ]:
def show_examples(bucket, n=25):
    ex = matched[matched["density_rescue_bucket"].eq(bucket)].copy()
    ex = ex.sort_values(["density_match_score", "density_score_gap"], ascending=False)
    display(ex[cols].head(n))

show_examples("grams_possible_but_still_missing_fdc", 25)
show_examples("count_density_name_match_only", 25)

## Optional local LLM density mapping

The fuzzy matching above is a baseline screen. This section optionally asks a local LM Studio model, such as Qwen, to choose the best density-table row from a shortlist for each target ingredient.

This is designed for the actual pipeline question: can we map the matched USDA/FDC food identity to a density entry and resolve grams from volume? It still writes only derived analysis files.

To run it:

1. Start LM Studio local server.
2. Load a Qwen instruct model.
3. Confirm the server is available at `http://localhost:1234/v1/chat/completions`.
4. Set `RUN_LOCAL_LLM = True` below.

Start with a small `LLM_LIMIT` and review outputs before scaling up.

In [ ]:
import json
import time
import urllib.error
import urllib.request

RUN_LOCAL_LLM = False
LM_STUDIO_URL = "http://localhost:1234/v1/chat/completions"
LM_STUDIO_MODEL = "local-model"
LLM_LIMIT = 30
LLM_TOP_K = 12
LLM_SLEEP_SEC = 0.1
LM_STUDIO_TIMEOUT_SEC = 300

def top_density_candidates(query, choices, k=12):
    if not query:
        return []
    if HAVE_RAPIDFUZZ:
        rows = process.extract(query, choices, scorer=fuzz.WRatio, limit=k)
        return [(idx, score) for _, score, idx in rows]
    scored = sorted(
        ((idx, _fallback_similarity(query, choice)) for idx, choice in enumerate(choices)),
        key=lambda x: x[1],
        reverse=True,
    )[:k]
    return scored

def density_candidate_payload(row, k=12):
    candidates = []
    for idx, score in top_density_candidates(row.get("match_query", ""), density_choices, k=k):
        d = density.iloc[int(idx)]
        candidates.append({
            "density_idx": int(idx),
            "score": round(float(score), 2),
            "food_group": d.get("food_group"),
            "food_name": d.get("food_name"),
            "density_g_per_ml": float(d.get("density_g_per_ml")),
            "density_g_per_ml_raw": str(d.get("density_g_per_ml_raw", d.get("density_g_per_ml"))),
        })
    return candidates

def extract_json_object(text):
    text = str(text).strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON object found: {text[:200]}")
    return json.loads(text[start:end + 1])

def ask_lm_studio(row, candidates):
    payload = {
        "model": LM_STUDIO_MODEL,
        "temperature": 0,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You map recipe/USDA food identities to food density table rows for volume-to-gram conversion. "
                    "Choose a density row only when it is the same food and physical state closely enough for bulk density. "
                    "Reject if the candidates differ materially, such as powder vs liquid, raw vs cooked when important, syrup vs fruit, or whole item vs chopped bulk. "
                    "Return JSON only."
                ),
            },
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "task": "Pick the best density candidate for resolving grams from a recipe volume amount.",
                        "recipe_row": {
                            "ingredient": row.get("ingredient"),
                            "parsed_quantity": row.get("quantity"),
                            "parsed_unit": row.get("unit"),
                            "parsed_name": row.get("name"),
                            "amount_kind_final": row.get("amount_kind_final"),
                            "llm_fdc_id": row.get("llm_fdc_id"),
                            "llm_description": row.get("llm_description"),
                            "grams_status": row.get("grams_status"),
                            "grams_method": row.get("grams_method"),
                        },
                        "density_candidates": candidates,
                        "return_schema": {
                            "selected_density_idx": "integer or null",
                            "use_density": "boolean",
                            "confidence": "high, medium, low, or reject",
                            "reason": "short explanation",
                            "warnings": ["short warning strings"],
                        },
                    },
                    ensure_ascii=False,
                ),
            },
        ],
    }
    req = urllib.request.Request(
        LM_STUDIO_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=LM_STUDIO_TIMEOUT_SEC) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    content = data["choices"][0]["message"]["content"]
    return extract_json_object(content)

def compute_density_grams_from_idx(row, density_idx):
    if pd.isna(density_idx):
        return np.nan
    ml = volume_ml(row.get("quantity_num"), row.get("unit"))
    if pd.isna(ml):
        return np.nan
    d = density.iloc[int(density_idx)]
    return float(ml) * float(d["density_g_per_ml"])

In [ ]:
llm_target = matched[
    matched["grams_status"].eq("no_portion")
    & matched["has_real_fdc"]
    & matched["amount_kind_final"].eq("volume")
    & matched["volume_ml"].notna()
].copy()

print(f"Local LLM target rows: {len(llm_target):,}")
display(llm_target[["ingredient", "quantity", "unit", "name", "llm_fdc_id", "llm_description", "food_name", "density_match_score", "density_rescue_bucket"]].head(25))

In [ ]:
llm_rows = []

if RUN_LOCAL_LLM:
    for original_idx, row in llm_target.head(LLM_LIMIT).iterrows():
        candidates = density_candidate_payload(row, k=LLM_TOP_K)
        try:
            decision = ask_lm_studio(row, candidates)
            error = None
        except (urllib.error.URLError, TimeoutError, ValueError, KeyError, json.JSONDecodeError) as exc:
            decision = {}
            error = repr(exc)

        selected_idx = decision.get("selected_density_idx")
        if selected_idx is not None:
            try:
                selected_idx = int(selected_idx)
            except (TypeError, ValueError):
                selected_idx = None

        use_density = bool(decision.get("use_density", False)) and selected_idx is not None
        grams_est = compute_density_grams_from_idx(row, selected_idx) if use_density else np.nan
        selected = density.iloc[selected_idx] if selected_idx is not None and 0 <= selected_idx < len(density) else pd.Series(dtype=object)

        llm_rows.append({
            "source_index": int(original_idx),
            "recipe_id": row.get("recipe_id"),
            "ingredient_idx": row.get("ingredient_idx"),
            "ingredient": row.get("ingredient"),
            "quantity": row.get("quantity"),
            "unit": row.get("unit"),
            "name": row.get("name"),
            "llm_fdc_id": row.get("llm_fdc_id"),
            "llm_description": row.get("llm_description"),
            "volume_ml": row.get("volume_ml"),
            "selected_density_idx": selected_idx,
            "selected_density_food_group": selected.get("food_group"),
            "selected_density_food_name": selected.get("food_name"),
            "selected_density_g_per_ml": selected.get("density_g_per_ml"),
            "local_llm_use_density": use_density,
            "local_llm_confidence": decision.get("confidence"),
            "local_llm_reason": decision.get("reason"),
            "local_llm_warnings": " | ".join(decision.get("warnings", []) or []),
            "local_llm_error": error,
            "local_llm_estimated_grams": grams_est,
            "candidate_json": json.dumps(candidates, ensure_ascii=False),
            "decision_json": json.dumps(decision, ensure_ascii=False),
        })
        time.sleep(LLM_SLEEP_SEC)

llm_density_matches = pd.DataFrame(llm_rows)

if RUN_LOCAL_LLM:
    display(llm_density_matches)
    print(llm_density_matches["local_llm_use_density"].value_counts(dropna=False).to_string())
else:
    print("RUN_LOCAL_LLM is False, so no LM Studio calls were made.")

## Reliability checks

Use these tables to look for false positives before proposing a pipeline change. Watch especially for:

- density rows that are a different state than the recipe ingredient, such as syrup vs powder, cooked vs raw, or juice vs whole fruit,
- low top-2 gaps, which suggest ambiguous density matches,
- very high or very low resulting gram estimates for ordinary kitchen quantities,
- rows where `missing_fdc` remains the blocker even if grams can be estimated.

In [ ]:
volume_candidates = matched[matched["density_rescue_bucket"].isin([
    "candidate_volume_rescue_likely",
    "candidate_volume_rescue_review",
])].copy()

display(volume_candidates["food_group"].value_counts().to_frame("n"))
display(volume_candidates.groupby("density_confidence")[["density_match_score", "density_score_gap", "density_estimated_grams"]].describe())

suspicious = volume_candidates[
    (volume_candidates["density_score_gap"] < 5)
    | (volume_candidates["density_estimated_grams"] <= 0)
    | (volume_candidates["density_estimated_grams"] > 2000)
].copy()
display(suspicious[cols].sort_values("density_match_score", ascending=False))

In [ ]:
matched.to_csv(OUT_DIR / "density_match_all_unresolved.csv", index=False)
likely_volume[cols].to_csv(OUT_DIR / "density_volume_rescue_likely.csv", index=False)
review_volume[cols].to_csv(OUT_DIR / "density_volume_rescue_review.csv", index=False)
missing_fdc_volume[cols].to_csv(OUT_DIR / "density_volume_possible_but_missing_fdc.csv", index=False)
count_name_only[cols].to_csv(OUT_DIR / "density_count_name_match_only.csv", index=False)

if "llm_density_matches" in globals() and not llm_density_matches.empty:
    llm_density_matches.to_csv(OUT_DIR / "density_lmstudio_volume_rescue.csv", index=False)

print(f"Wrote review artifacts to {OUT_DIR}")

## Interpretation template

After running the notebook, use this language as the starting point for the group update:

- Density is most promising for `no_portion` + real-FDC + volume rows because nutrition identity is already known and only volume-to-gram conversion is missing.
- Density does not solve `missing_fdc` by itself; those rows still need an FDC id before they can enter the nutrient pipeline.
- Density is not generally enough for count rows because counts require item size or container mass, not bulk g/ml density.
- A safe pipeline augmentation would add a density fallback after USDA volume portions fail, gated by a curated density match or a high-confidence mapping table rather than unconstrained fuzzy matching at runtime.